# 03 - Regresión: Predicción de Satisfacción

**Pregunta de negocio:** ¿Qué impulsa la satisfacción del cliente?

## Objetivos
- Predecir `satisfaction_score` (escala ordinal 1-5) usando regresión
- Identificar los factores demográficos y de uso del vehículo que más influyen en la satisfacción
- Comparar modelos: Regresión Lineal, Random Forest y XGBoost
- Extraer insights accionables para mejorar la experiencia del cliente

## Enfoque: ¿Regresión o Clasificación?

Aunque `satisfaction_score` es técnicamente una variable ordinal (1-5), podemos tratarla como **regresión** porque:
1. La escala es **ordenada y equiespaciada** (1 < 2 < 3 < 4 < 5)
2. La diferencia entre 3→4 es conceptualmente similar a 4→5
3. La regresión nos permite obtener **importancia de features interpretable** (coeficientes, feature importance)
4. Podemos redondear/clipear las predicciones al rango [1, 5]

La ventaja clave: los **coeficientes** de regresión lineal nos dicen directamente *cuánto* y *en qué dirección* cada variable afecta la satisfacción.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
data_dir = os.path.join(project_root, "data/raw")
processed_dir = os.path.join(project_root, "data/processed")
rng = np.random.default_rng(42)

vtype_colors = {'electrico': '#2ecc71', 'gasolina': '#3498db', 'hibrido': '#9b59b6', 'deportivo': '#e74c3c'}

print("Librerías cargadas correctamente")

## 1. Carga y exploración de datos

Cargamos el dataset fusionado que combina telemetría agregada por vehículo con datos demográficos de las encuestas. Nuestro target es `satisfaction_score` (1-5).

In [ ]:
# Cargar datos fusionados (telemetría + encuestas)
features_path = os.path.join(processed_dir, "features_ml.csv")
merged_path = os.path.join(processed_dir, "vehicle_survey_merged.csv")

if os.path.exists(features_path):
    df = pd.read_csv(features_path)
    print(f"Cargado desde features_ml.csv")
else:
    df = pd.read_csv(merged_path)
    print(f"Cargado desde vehicle_survey_merged.csv")

# Cargar encuestas originales para referencia
surveys = pd.read_csv(os.path.join(data_dir, "surveys/buyer_surveys.csv"))

print(f"\nDimensiones: {df.shape}")
print(f"Registros:   {len(df):,}")
print(f"Columnas:    {df.shape[1]}")
print(f"\nColumnas disponibles:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col:<30s} → {df[col].dtype}")

print(f"\nValores nulos por columna:")
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else "  Sin valores nulos")

In [ ]:
# Distribución del target: satisfaction_score
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribución general
ax = axes[0]
counts = df['satisfaction_score'].value_counts().sort_index()
colors_sat = plt.cm.RdYlGn(np.linspace(0.2, 0.9, 5))
bars = ax.bar(counts.index, counts.values, color=colors_sat, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Puntuación de Satisfacción', fontsize=12)
ax.set_ylabel('Número de clientes', fontsize=12)
ax.set_title('Distribución de Satisfacción', fontsize=13, fontweight='bold')
ax.set_xticks([1, 2, 3, 4, 5])

# 2. Distribución por tipo de vehículo
ax = axes[1]
sat_by_type = df.groupby('vehicle_type')['satisfaction_score'].value_counts().unstack(fill_value=0)
sat_by_type = sat_by_type.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)
sat_by_type.plot(kind='bar', ax=ax, color=colors_sat, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Tipo de vehículo', fontsize=12)
ax.set_ylabel('Número de clientes', fontsize=12)
ax.set_title('Satisfacción por Tipo de Vehículo', fontsize=13, fontweight='bold')
ax.legend(title='Score', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)

# 3. Boxplot de satisfacción por tipo
ax = axes[2]
for i, (vtype, color) in enumerate(vtype_colors.items()):
    subset = df[df['vehicle_type'] == vtype]['satisfaction_score']
    if len(subset) > 0:
        bp = ax.boxplot(subset, positions=[i], widths=0.6,
                       patch_artist=True, boxprops=dict(facecolor=color, alpha=0.7),
                       medianprops=dict(color='black', linewidth=2))
ax.set_xticks(range(len(vtype_colors)))
ax.set_xticklabels([v.capitalize() for v in vtype_colors.keys()])
ax.set_ylabel('Satisfacción', fontsize=12)
ax.set_title('Distribución por Tipo', fontsize=13, fontweight='bold')

plt.suptitle('Análisis del Target: satisfaction_score', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print(f"\nEstadísticas del target:")
print(f"  Media:    {df['satisfaction_score'].mean():.2f}")
print(f"  Mediana:  {df['satisfaction_score'].median():.1f}")
print(f"  Std:      {df['satisfaction_score'].std():.2f}")
print(f"  Min-Max:  {df['satisfaction_score'].min()} - {df['satisfaction_score'].max()}")
print(f"\nBalance de clases:")
for score in sorted(df['satisfaction_score'].unique()):
    n = (df['satisfaction_score'] == score).sum()
    print(f"  Score {score}: {n:4d} ({n/len(df)*100:5.1f}%)")

## 2. Preparación de features

Seleccionamos y transformamos las variables predictoras en tres categorías:

1. **Demográficas**: edad, género, nivel de ingresos (codificados)
2. **Uso del vehículo**: velocidad media, consumo, km totales, frenadas bruscas, estilo de conducción
3. **Tipo de vehículo**: codificado con dummies

Las variables categóricas se codifican numéricamente (Label Encoding para ordinales, One-Hot para nominales).

In [ ]:
# Filtrar solo registros con encuesta (has_survey == True si existe la columna)
if 'has_survey' in df.columns:
    df_model = df[df['has_survey'] == True].copy()
    print(f"Registros con encuesta: {len(df_model)}")
else:
    df_model = df.dropna(subset=['satisfaction_score']).copy()
    print(f"Registros con satisfaction_score: {len(df_model)}")

# --- Codificación de variables categóricas ---

# 1. income_bracket: codificación ordinal (bajo → alto)
income_order = {'bajo': 1, 'medio-bajo': 2, 'medio': 3, 'medio-alto': 4, 'alto': 5}
df_model['income_encoded'] = df_model['income_bracket'].map(income_order)
print(f"\nincome_bracket codificado:")
for k, v in income_order.items():
    print(f"  {k:<12s} → {v}")

# 2. gender: codificación binaria
df_model['gender_encoded'] = (df_model['gender'] == 'masculino').astype(int)
print(f"\ngender codificado: masculino=1, femenino=0")

# 3. driving_style: codificación ordinal
style_order = {'calm': 0, 'normal': 1, 'aggressive': 2}
df_model['driving_style_encoded'] = df_model['driving_style'].map(style_order)
print(f"\ndriving_style codificado:")
for k, v in style_order.items():
    print(f"  {k:<12s} → {v}")

# 4. vehicle_type: one-hot encoding
vtype_dummies = pd.get_dummies(df_model['vehicle_type'], prefix='vtype', drop_first=False)
df_model = pd.concat([df_model, vtype_dummies], axis=1)
print(f"\nvehicle_type → dummies: {list(vtype_dummies.columns)}")

# --- Selección de features ---
feature_cols = [
    # Demográficas
    'age', 'gender_encoded', 'income_encoded',
    # Uso del vehículo (telemetría)
    'speed_mean', 'consumption_mean', 'estimated_km', 'harsh_braking_count',
    'driving_style_encoded',
    # Tipo de vehículo (dummies)
] + [c for c in vtype_dummies.columns]

# Verificar que todas existen
available_features = [c for c in feature_cols if c in df_model.columns]
missing_features = [c for c in feature_cols if c not in df_model.columns]
if missing_features:
    print(f"\nFeatures no encontradas (se omiten): {missing_features}")

# Si km_driven existe y estimated_km no, usar km_driven
if 'estimated_km' not in df_model.columns and 'km_driven' in df_model.columns:
    df_model['estimated_km'] = df_model['km_driven']
    if 'estimated_km' not in available_features:
        available_features.append('estimated_km')
    print("Usando km_driven como estimated_km")

target_col = 'satisfaction_score'

print(f"\n{'='*50}")
print(f"Features seleccionadas ({len(available_features)}):")
for f in available_features:
    print(f"  - {f}")
print(f"Target: {target_col}")

In [ ]:
# Train/Test split
X = df_model[available_features].copy()
y = df_model[target_col].copy()

# Eliminar filas con NaN en features o target
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

print(f"Datos limpios: {len(X)} registros")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} registros")
print(f"Test:  {len(X_test)} registros")

# Escalar features para regresión lineal
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=available_features, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=available_features, index=X_test.index
)

# Verificar distribución del target en train y test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, data) in zip(axes, [('Train', y_train), ('Test', y_test)]):
    vc = data.value_counts().sort_index()
    ax.bar(vc.index, vc.values, color=colors_sat, edgecolor='white')
    ax.set_title(f'{name} (n={len(data)})', fontsize=13, fontweight='bold')
    ax.set_xlabel('Satisfacción')
    ax.set_ylabel('Frecuencia')
    ax.set_xticks([1, 2, 3, 4, 5])
plt.suptitle('Distribución del target: Train vs Test (split estratificado)',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 3. Regresión Lineal (baseline)

El modelo más simple e interpretable. Los coeficientes nos dicen directamente:
- **Signo**: ¿la variable aumenta (+) o disminuye (-) la satisfacción?
- **Magnitud** (con features escaladas): ¿cuánto impacta cada variable relativa a las demás?

Como el target está en [1, 5], clipeamos las predicciones a ese rango.

In [ ]:
# --- Regresión Lineal ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predicciones (clipeadas a [1, 5])
y_pred_lr_train = np.clip(lr.predict(X_train_scaled), 1, 5)
y_pred_lr_test = np.clip(lr.predict(X_test_scaled), 1, 5)

# Métricas
def eval_regression(y_true, y_pred, name="Modelo"):
    """Evalúa un modelo de regresión con múltiples métricas."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    # Accuracy si redondeamos al entero más cercano
    y_rounded = np.clip(np.round(y_pred), 1, 5).astype(int)
    accuracy = (y_rounded == y_true).mean()
    return {'Modelo': name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'Accuracy (redondeado)': accuracy}

lr_train_metrics = eval_regression(y_train, y_pred_lr_train, "LR (train)")
lr_test_metrics = eval_regression(y_test, y_pred_lr_test, "LR (test)")

print("=== Regresión Lineal ===")
print(f"\n{'Métrica':<25s} {'Train':>10s} {'Test':>10s}")
print("-" * 47)
for metric in ['RMSE', 'MAE', 'R²', 'Accuracy (redondeado)']:
    print(f"{metric:<25s} {lr_train_metrics[metric]:>10.4f} {lr_test_metrics[metric]:>10.4f}")

print(f"\nIntercepto: {lr.intercept_:.4f}")

In [ ]:
# Coeficientes de la regresión lineal (features estandarizadas → comparables)
coef_df = pd.DataFrame({
    'Feature': available_features,
    'Coeficiente': lr.coef_
}).sort_values('Coeficiente', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(available_features) * 0.5)))

colors_coef = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coeficiente']]
bars = ax.barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors_coef, alpha=0.8,
               edgecolor='white', linewidth=0.5)

# Anotar valores
for bar, val in zip(bars, coef_df['Coeficiente']):
    offset = 0.005 * np.sign(val)
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center', fontsize=9, fontweight='bold')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente estandarizado', fontsize=12)
ax.set_title('Regresión Lineal: ¿Qué aumenta y qué disminuye la satisfacción?\n'
             '(coeficientes estandarizados - magnitud comparable entre features)',
             fontsize=13, fontweight='bold')

# Leyenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Aumenta satisfacción'),
                   Patch(facecolor='#e74c3c', label='Disminuye satisfacción')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

print("Interpretación de coeficientes (features estandarizadas):")
print("  - Un coeficiente de +0.1 significa: al aumentar la feature 1 desviación estándar,")
print("    la satisfacción predicha sube 0.1 puntos")
print("  - Signo positivo → más de esa variable = más satisfacción")
print("  - Signo negativo → más de esa variable = menos satisfacción")

## 4. Random Forest Regressor

El Random Forest captura relaciones **no lineales** e **interacciones** entre features que la regresión lineal no puede modelar. Además, proporciona una medida de importancia de features basada en la reducción de impureza (Gini).

In [ ]:
# --- Random Forest (no necesita escalado) ---
# Tuning manual de hiperparámetros con validación cruzada
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [5, 10, 20],
}

# Búsqueda simplificada: probar combinaciones clave
best_score = -np.inf
best_params = {}
results_rf = []

print("Tuning Random Forest (búsqueda de hiperparámetros)...\n")
for n_est in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        for min_leaf in param_grid['min_samples_leaf']:
            rf_temp = RandomForestRegressor(
                n_estimators=n_est, max_depth=depth, min_samples_leaf=min_leaf,
                random_state=42, n_jobs=-1
            )
            cv_scores = cross_val_score(rf_temp, X_train, y_train, cv=5,
                                        scoring='neg_root_mean_squared_error')
            mean_rmse = -cv_scores.mean()
            results_rf.append({
                'n_estimators': n_est, 'max_depth': depth,
                'min_samples_leaf': min_leaf, 'CV_RMSE': mean_rmse
            })
            if -mean_rmse > best_score:
                best_score = -mean_rmse
                best_params = {'n_estimators': n_est, 'max_depth': depth,
                              'min_samples_leaf': min_leaf}

results_rf_df = pd.DataFrame(results_rf).sort_values('CV_RMSE')
print("Top 5 combinaciones:")
print(results_rf_df.head().to_string(index=False))
print(f"\nMejores parámetros: {best_params}")

# Entrenar con mejores parámetros
rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf_train = np.clip(rf.predict(X_train), 1, 5)
y_pred_rf_test = np.clip(rf.predict(X_test), 1, 5)

rf_train_metrics = eval_regression(y_train, y_pred_rf_train, "RF (train)")
rf_test_metrics = eval_regression(y_test, y_pred_rf_test, "RF (test)")

print(f"\n{'Métrica':<25s} {'Train':>10s} {'Test':>10s}")
print("-" * 47)
for metric in ['RMSE', 'MAE', 'R²', 'Accuracy (redondeado)']:
    print(f"{metric:<25s} {rf_train_metrics[metric]:>10.4f} {rf_test_metrics[metric]:>10.4f}")

In [ ]:
# Feature importance: Random Forest vs Regresión Lineal
rf_importance = pd.DataFrame({
    'Feature': available_features,
    'RF_Importance': rf.feature_importances_,
    'LR_Coef_Abs': np.abs(lr.coef_)
}).sort_values('RF_Importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(available_features) * 0.5)))

# Random Forest importance
ax = axes[0]
ax.barh(rf_importance['Feature'], rf_importance['RF_Importance'],
        color='#3498db', alpha=0.8, edgecolor='white')
for i, (_, row) in enumerate(rf_importance.iterrows()):
    ax.text(row['RF_Importance'] + 0.002, i, f"{row['RF_Importance']:.3f}",
            va='center', fontsize=9)
ax.set_xlabel('Importancia (reducción de impureza)', fontsize=11)
ax.set_title('Random Forest: Feature Importance', fontsize=13, fontweight='bold')

# Regresión Lineal coeficientes absolutos
ax = axes[1]
lr_sorted = rf_importance.sort_values('LR_Coef_Abs', ascending=True)
ax.barh(lr_sorted['Feature'], lr_sorted['LR_Coef_Abs'],
        color='#e74c3c', alpha=0.8, edgecolor='white')
for i, (_, row) in enumerate(lr_sorted.iterrows()):
    ax.text(row['LR_Coef_Abs'] + 0.002, i, f"{row['LR_Coef_Abs']:.3f}",
            va='center', fontsize=9)
ax.set_xlabel('|Coeficiente estandarizado|', fontsize=11)
ax.set_title('Regresión Lineal: |Coeficientes|', fontsize=13, fontweight='bold')

plt.suptitle('Comparación de Importancia de Features: RF vs LR',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("Nota: Las dos medidas de importancia pueden diferir porque:")
print("  - RF captura relaciones no lineales que LR no ve")
print("  - LR mide efecto marginal lineal, RF mide capacidad de separación")

## 5. XGBoost (Gradient Boosting)

XGBoost construye árboles secuencialmente, donde cada nuevo árbol corrige los errores del anterior. Suele ser el modelo más preciso para datos tabulares. Usamos **permutation importance** de sklearn como alternativa robusta a SHAP para medir la importancia real de cada feature.

In [ ]:
# --- Gradient Boosting (XGBoost-style con sklearn) ---
# Intentamos importar xgboost; si no está disponible, usamos GradientBoostingRegressor de sklearn
try:
    from xgboost import XGBRegressor
    xgb_model = XGBRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    )
    model_name = "XGBoost"
    print("Usando XGBRegressor (xgboost instalado)")
except ImportError:
    xgb_model = GradientBoostingRegressor(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        subsample=0.8, random_state=42
    )
    model_name = "GradientBoosting"
    print("xgboost no disponible, usando GradientBoostingRegressor de sklearn")

xgb_model.fit(X_train, y_train)

y_pred_xgb_train = np.clip(xgb_model.predict(X_train), 1, 5)
y_pred_xgb_test = np.clip(xgb_model.predict(X_test), 1, 5)

xgb_train_metrics = eval_regression(y_train, y_pred_xgb_train, f"{model_name} (train)")
xgb_test_metrics = eval_regression(y_test, y_pred_xgb_test, f"{model_name} (test)")

print(f"\n=== {model_name} ===")
print(f"\n{'Métrica':<25s} {'Train':>10s} {'Test':>10s}")
print("-" * 47)
for metric in ['RMSE', 'MAE', 'R²', 'Accuracy (redondeado)']:
    print(f"{metric:<25s} {xgb_train_metrics[metric]:>10.4f} {xgb_test_metrics[metric]:>10.4f}")

In [ ]:
# Permutation Importance (más robusto que feature_importances_ basado en impureza)
# Mide cuánto empeora el modelo al permutar aleatoriamente cada feature
perm_imp = permutation_importance(
    xgb_model, X_test, y_test, n_repeats=20, random_state=42, n_jobs=-1,
    scoring='neg_root_mean_squared_error'
)

perm_df = pd.DataFrame({
    'Feature': available_features,
    'Importance_Mean': perm_imp.importances_mean,
    'Importance_Std': perm_imp.importances_std
}).sort_values('Importance_Mean', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(available_features) * 0.5)))

colors_perm = plt.cm.viridis(np.linspace(0.2, 0.9, len(perm_df)))
ax.barh(perm_df['Feature'], perm_df['Importance_Mean'],
        xerr=perm_df['Importance_Std'], color=colors_perm, alpha=0.8,
        edgecolor='white', linewidth=0.5, capsize=3)

for i, (_, row) in enumerate(perm_df.iterrows()):
    ax.text(row['Importance_Mean'] + row['Importance_Std'] + 0.001, i,
            f"{row['Importance_Mean']:.4f}", va='center', fontsize=9)

ax.set_xlabel('Aumento en RMSE al permutar (mayor = más importante)', fontsize=11)
ax.set_title(f'{model_name}: Permutation Importance\n'
             f'(mide cuánto empeora el modelo sin cada feature)',
             fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("Interpretación:")
print("  - Features con alta permutation importance son ESENCIALES para la predicción")
print("  - Features cercanas a 0 podrían eliminarse sin afectar el rendimiento")
print("  - Las barras de error muestran la variabilidad entre repeticiones")

## 6. Análisis de Key Drivers: Partial Dependence Plots

Los Partial Dependence Plots (PDP) muestran cómo cambia la predicción promedio al variar **una sola feature**, manteniendo las demás constantes. Nos permiten responder: *"¿Cómo afecta la edad/km/ingreso a la satisfacción predicha?"*

In [ ]:
# Identificar las top 3 features por permutation importance
top3_features = perm_df.sort_values('Importance_Mean', ascending=False).head(3)['Feature'].tolist()
top3_indices = [available_features.index(f) for f in top3_features]

print(f"Top 3 features para Partial Dependence Plots: {top3_features}\n")

# Partial Dependence Plots usando el mejor modelo de árboles
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat, feat_idx in zip(axes, top3_features, top3_indices):
    # Calcular PDP manualmente para mayor control visual
    feat_values = np.linspace(
        X_train[feat].quantile(0.05),
        X_train[feat].quantile(0.95),
        50
    )
    
    pdp_values = []
    for val in feat_values:
        X_temp = X_train.copy()
        X_temp[feat] = val
        preds = xgb_model.predict(X_temp)
        pdp_values.append(preds.mean())
    
    ax.plot(feat_values, pdp_values, color='#2c3e50', linewidth=2.5)
    ax.fill_between(feat_values, pdp_values, alpha=0.1, color='#3498db')
    
    # Distribución de la feature (histograma en eje secundario)
    ax2 = ax.twinx()
    ax2.hist(X_train[feat], bins=30, alpha=0.15, color='gray', density=True)
    ax2.set_ylabel('Densidad de datos', fontsize=9, color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    
    ax.set_xlabel(feat, fontsize=12)
    ax.set_ylabel('Satisfacción predicha (promedio)', fontsize=11)
    ax.set_title(f'Partial Dependence: {feat}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Partial Dependence Plots ({model_name})\n'
             f'¿Cómo afecta cada feature a la satisfacción predicha?',
             fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Lectura de los PDP:")
print("  - Línea ascendente → mayor valor de la feature = mayor satisfacción")
print("  - Línea descendente → mayor valor de la feature = menor satisfacción")
print("  - Línea plana → la feature no tiene efecto marginal")
print("  - Curvas no lineales → el efecto depende del rango de la feature")

## 7. Comparación de modelos

Evaluamos los tres modelos con validación cruzada (5-fold) para obtener estimaciones robustas del rendimiento. Además, visualizamos las predicciones vs valores reales del mejor modelo.

In [ ]:
# Validación cruzada para los tres modelos
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    'Regresión Lineal': LinearRegression(),
    'Random Forest': RandomForestRegressor(**best_params, random_state=42, n_jobs=-1),
    f'{model_name}': xgb_model.__class__(**xgb_model.get_params()),
}

cv_results = []
print("Validación cruzada (5-fold):\n")
print(f"{'Modelo':<25s} {'RMSE (media)':>12s} {'RMSE (std)':>12s} {'MAE (media)':>12s} {'R² (media)':>12s}")
print("-" * 75)

for name, model in models_cv.items():
    # Para LR, usar datos escalados
    if name == 'Regresión Lineal':
        X_cv = pd.DataFrame(scaler.fit_transform(X), columns=available_features)
    else:
        X_cv = X
    
    rmse_scores = -cross_val_score(model, X_cv, y, cv=kf, scoring='neg_root_mean_squared_error')
    mae_scores = -cross_val_score(model, X_cv, y, cv=kf, scoring='neg_mean_absolute_error')
    r2_scores = cross_val_score(model, X_cv, y, cv=kf, scoring='r2')
    
    cv_results.append({
        'Modelo': name,
        'RMSE_mean': rmse_scores.mean(), 'RMSE_std': rmse_scores.std(),
        'MAE_mean': mae_scores.mean(), 'MAE_std': mae_scores.std(),
        'R2_mean': r2_scores.mean(), 'R2_std': r2_scores.std(),
    })
    
    print(f"{name:<25s} {rmse_scores.mean():>10.4f} ± {rmse_scores.std():.4f} "
          f"{mae_scores.mean():>10.4f} ± {mae_scores.std():.4f} "
          f"{r2_scores.mean():>10.4f} ± {r2_scores.std():.4f}")

cv_df = pd.DataFrame(cv_results)

# Visualizar comparación
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics_plot = [('RMSE_mean', 'RMSE_std', 'RMSE (menor = mejor)', True),
                ('MAE_mean', 'MAE_std', 'MAE (menor = mejor)', True),
                ('R2_mean', 'R2_std', 'R² (mayor = mejor)', False)]

model_colors = ['#3498db', '#2ecc71', '#e74c3c']

for ax, (mean_col, std_col, title, invert) in zip(axes, metrics_plot):
    bars = ax.bar(cv_df['Modelo'], cv_df[mean_col], yerr=cv_df[std_col],
                  color=model_colors, alpha=0.8, edgecolor='white', capsize=5)
    for bar, val in zip(bars, cv_df[mean_col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + cv_df[std_col].max() * 0.1,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(mean_col.replace('_mean', ''))
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Comparación de Modelos (Validación Cruzada 5-Fold)',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicho: scatter plot del mejor modelo (test set)
# Determinar el mejor modelo por RMSE en test
all_test_metrics = {
    'Regresión Lineal': lr_test_metrics,
    'Random Forest': rf_test_metrics,
    f'{model_name}': xgb_test_metrics,
}

best_model_name = min(all_test_metrics, key=lambda k: all_test_metrics[k]['RMSE'])
print(f"Mejor modelo por RMSE en test: {best_model_name}")

# Seleccionar predicciones del mejor modelo
best_preds = {
    'Regresión Lineal': y_pred_lr_test,
    'Random Forest': y_pred_rf_test,
    f'{model_name}': y_pred_xgb_test,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, best_preds.items()):
    # Scatter con jitter para visualizar puntos superpuestos
    jitter_x = y_test + np.random.normal(0, 0.08, len(y_test))
    jitter_y = y_pred + np.random.normal(0, 0.08, len(y_pred))
    
    ax.scatter(jitter_x, jitter_y, alpha=0.3, s=20, color='steelblue')
    ax.plot([0.5, 5.5], [0.5, 5.5], 'r--', linewidth=2, label='Predicción perfecta')
    
    metrics = all_test_metrics[name]
    ax.set_xlabel('Satisfacción real', fontsize=12)
    ax.set_ylabel('Satisfacción predicha', fontsize=12)
    ax.set_title(f'{name}\nRMSE={metrics["RMSE"]:.3f}, R²={metrics["R²"]:.3f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlim(0.5, 5.5)
    ax.set_ylim(0.5, 5.5)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.legend(fontsize=9)
    ax.set_aspect('equal')

plt.suptitle('Actual vs Predicho (Test Set) - Todos los modelos',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Distribución de residuos del mejor modelo
best_residuals = y_test.values - best_preds[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(best_residuals, bins=30, color='#3498db', alpha=0.7, edgecolor='white', density=True)
ax.axvline(0, color='red', linewidth=2, linestyle='--')
ax.set_xlabel('Residuo (real - predicho)', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.set_title(f'Distribución de Residuos ({best_model_name})', fontsize=13, fontweight='bold')
ax.text(0.05, 0.95, f'Media: {best_residuals.mean():.3f}\nStd: {best_residuals.std():.3f}',
        transform=ax.transAxes, va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax = axes[1]
ax.scatter(best_preds[best_model_name], best_residuals, alpha=0.3, s=20, color='steelblue')
ax.axhline(0, color='red', linewidth=2, linestyle='--')
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Residuo', fontsize=12)
ax.set_title(f'Residuos vs Predicciones ({best_model_name})', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Insights de negocio: ¿Qué impulsa la satisfacción?

Combinamos los resultados de los tres modelos para generar visualizaciones accionables:
1. **Ranking de drivers de satisfacción**: consenso entre modelos
2. **Análisis por segmento**: ¿qué grupos de clientes están menos satisfechos y por qué?
3. **Recomendaciones concretas** para el negocio

In [ ]:
# --- Ranking consolidado de drivers de satisfacción ---
# Normalizar importancias de cada modelo al rango [0, 1]
from sklearn.preprocessing import MinMaxScaler

drivers_df = pd.DataFrame({'Feature': available_features})

# LR: coeficientes absolutos estandarizados
drivers_df['LR_importance'] = np.abs(lr.coef_)

# RF: feature importance
drivers_df['RF_importance'] = rf.feature_importances_

# Permutation importance del mejor modelo de boosting
drivers_df['Perm_importance'] = perm_imp.importances_mean

# Normalizar cada columna a [0, 1]
for col in ['LR_importance', 'RF_importance', 'Perm_importance']:
    vals = drivers_df[col].values.reshape(-1, 1)
    drivers_df[col + '_norm'] = MinMaxScaler().fit_transform(vals)

# Score combinado (promedio de las tres importancias normalizadas)
drivers_df['Combined_Score'] = drivers_df[['LR_importance_norm', 'RF_importance_norm',
                                            'Perm_importance_norm']].mean(axis=1)
drivers_df = drivers_df.sort_values('Combined_Score', ascending=True)

# Etiquetas legibles
label_map = {
    'age': 'Edad', 'gender_encoded': 'Género (masc.)', 'income_encoded': 'Nivel de ingreso',
    'speed_mean': 'Velocidad media', 'consumption_mean': 'Consumo medio',
    'estimated_km': 'Km recorridos', 'harsh_braking_count': 'Frenadas bruscas',
    'driving_style_encoded': 'Estilo de conducción',
    'vtype_electrico': 'Tipo: Eléctrico', 'vtype_gasolina': 'Tipo: Gasolina',
    'vtype_hibrido': 'Tipo: Híbrido', 'vtype_deportivo': 'Tipo: Deportivo',
}
drivers_df['Label'] = drivers_df['Feature'].map(label_map).fillna(drivers_df['Feature'])

# Visualización: Ranking de drivers
fig, ax = plt.subplots(figsize=(12, max(6, len(available_features) * 0.55)))

colors_rank = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(drivers_df)))
bars = ax.barh(drivers_df['Label'], drivers_df['Combined_Score'],
               color=colors_rank, edgecolor='white', linewidth=0.5, alpha=0.9)

for bar, val in zip(bars, drivers_df['Combined_Score']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Importancia combinada (promedio normalizado de 3 modelos)', fontsize=12)
ax.set_title('Drivers de Satisfacción del Cliente\n'
             '(Consenso: Regresión Lineal + Random Forest + Permutation Importance)',
             fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Top 5 drivers de satisfacción (mayor impacto en la predicción):")
top5 = drivers_df.sort_values('Combined_Score', ascending=False).head(5)
for i, (_, row) in enumerate(top5.iterrows(), 1):
    direction = "+" if lr.coef_[available_features.index(row['Feature'])] > 0 else "-"
    print(f"  {i}. {row['Label']:<25s} (score: {row['Combined_Score']:.3f}, efecto LR: {direction})")

In [ ]:
# --- Análisis por segmento: ¿Quiénes son los clientes menos satisfechos? ---

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Satisfacción por tipo de vehículo
ax = axes[0, 0]
sat_vtype = df_model.groupby('vehicle_type')['satisfaction_score'].agg(['mean', 'std', 'count'])
sat_vtype = sat_vtype.sort_values('mean')
colors_bar = [vtype_colors.get(v, '#999999') for v in sat_vtype.index]
bars = ax.bar(sat_vtype.index.str.capitalize(), sat_vtype['mean'],
              yerr=sat_vtype['std'], color=colors_bar, alpha=0.8,
              edgecolor='white', capsize=5)
for bar, (_, row) in zip(bars, sat_vtype.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + row['std'] + 0.05,
            f'{row["mean"]:.2f}\n(n={int(row["count"])})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Satisfacción media', fontsize=12)
ax.set_title('Satisfacción por Tipo de Vehículo', fontsize=13, fontweight='bold')
ax.set_ylim(0, 6)

# 2. Satisfacción por nivel de ingreso
ax = axes[0, 1]
sat_income = df_model.groupby('income_bracket')['satisfaction_score'].agg(['mean', 'std', 'count'])
income_order_list = ['bajo', 'medio-bajo', 'medio', 'medio-alto', 'alto']
sat_income = sat_income.reindex([i for i in income_order_list if i in sat_income.index])
colors_income = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(sat_income)))
bars = ax.bar(sat_income.index, sat_income['mean'], yerr=sat_income['std'],
              color=colors_income, alpha=0.8, edgecolor='white', capsize=5)
for bar, (_, row) in zip(bars, sat_income.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + row['std'] + 0.05,
            f'{row["mean"]:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Satisfacción media', fontsize=12)
ax.set_title('Satisfacción por Nivel de Ingreso', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=15)
ax.set_ylim(0, 6)

# 3. Satisfacción por estilo de conducción
ax = axes[1, 0]
sat_style = df_model.groupby('driving_style')['satisfaction_score'].agg(['mean', 'std', 'count'])
colors_style = ['#2ecc71', '#f1c40f', '#e74c3c']
style_order_list = ['calm', 'normal', 'aggressive']
sat_style = sat_style.reindex([s for s in style_order_list if s in sat_style.index])
bars = ax.bar(sat_style.index.str.capitalize(), sat_style['mean'], yerr=sat_style['std'],
              color=colors_style[:len(sat_style)], alpha=0.8, edgecolor='white', capsize=5)
for bar, (_, row) in zip(bars, sat_style.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + row['std'] + 0.05,
            f'{row["mean"]:.2f}\n(n={int(row["count"])})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Satisfacción media', fontsize=12)
ax.set_title('Satisfacción por Estilo de Conducción', fontsize=13, fontweight='bold')
ax.set_ylim(0, 6)

# 4. Satisfacción por grupo de edad
ax = axes[1, 1]
df_model['age_group'] = pd.cut(df_model['age'], bins=[18, 30, 40, 50, 60, 100],
                                labels=['18-30', '31-40', '41-50', '51-60', '60+'])
sat_age = df_model.groupby('age_group', observed=True)['satisfaction_score'].agg(['mean', 'std', 'count'])
colors_age = plt.cm.coolwarm(np.linspace(0.2, 0.8, len(sat_age)))
bars = ax.bar(sat_age.index.astype(str), sat_age['mean'], yerr=sat_age['std'],
              color=colors_age, alpha=0.8, edgecolor='white', capsize=5)
for bar, (_, row) in zip(bars, sat_age.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + row['std'] + 0.05,
            f'{row["mean"]:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Satisfacción media', fontsize=12)
ax.set_xlabel('Grupo de edad', fontsize=12)
ax.set_title('Satisfacción por Grupo de Edad', fontsize=13, fontweight='bold')
ax.set_ylim(0, 6)

plt.suptitle('Análisis de Segmentación: ¿Qué grupos están menos satisfechos?',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap cruzado: satisfacción promedio por tipo de vehículo x ingreso ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap: vehicle_type x income_bracket
ax = axes[0]
pivot_sat = df_model.pivot_table(values='satisfaction_score', index='vehicle_type',
                                  columns='income_bracket', aggfunc='mean')
# Reordenar columnas
pivot_sat = pivot_sat.reindex(columns=[i for i in income_order_list if i in pivot_sat.columns])
sns.heatmap(pivot_sat, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax,
            vmin=1, vmax=5, cbar_kws={'label': 'Satisfacción media'},
            linewidths=1, linecolor='white')
ax.set_title('Satisfacción: Tipo de Vehículo x Nivel de Ingreso',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Tipo de vehículo')
ax.set_xlabel('Nivel de ingreso')

# Heatmap: vehicle_type x driving_style
ax = axes[1]
pivot_sat2 = df_model.pivot_table(values='satisfaction_score', index='vehicle_type',
                                   columns='driving_style', aggfunc='mean')
pivot_sat2 = pivot_sat2.reindex(columns=[s for s in style_order_list if s in pivot_sat2.columns])
sns.heatmap(pivot_sat2, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax,
            vmin=1, vmax=5, cbar_kws={'label': 'Satisfacción media'},
            linewidths=1, linecolor='white')
ax.set_title('Satisfacción: Tipo de Vehículo x Estilo de Conducción',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Tipo de vehículo')
ax.set_xlabel('Estilo de conducción')

plt.suptitle('Mapas de Calor: Satisfacción por Segmentos Cruzados',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("Estos mapas de calor revelan qué COMBINACIONES de segmentos tienen mayor/menor satisfacción.")
print("Los segmentos en rojo/naranja son oportunidades prioritarias de mejora.")

In [ ]:
# --- Dashboard resumen: Drivers de satisfacción con dirección del efecto ---
# Combinar importancia con la dirección (signo del coeficiente LR)

fig, ax = plt.subplots(figsize=(14, max(7, len(available_features) * 0.6)))

drivers_sorted = drivers_df.sort_values('Combined_Score', ascending=False).copy()
drivers_sorted['LR_coef'] = [lr.coef_[available_features.index(f)] for f in drivers_sorted['Feature']]
drivers_sorted['Direction'] = drivers_sorted['LR_coef'].apply(
    lambda x: 'Aumenta satisfacción' if x > 0 else 'Disminuye satisfacción'
)

# Crear barras con la importancia, coloreadas por dirección
y_pos = range(len(drivers_sorted))
colors_dir = ['#27ae60' if d == 'Aumenta satisfacción' else '#c0392b'
              for d in drivers_sorted['Direction']]

bars = ax.barh(list(y_pos), drivers_sorted['Combined_Score'].values,
               color=colors_dir, alpha=0.85, edgecolor='white', linewidth=0.5)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(drivers_sorted['Label'].values, fontsize=11)
ax.invert_yaxis()

# Anotar importancia y coeficiente
for i, (_, row) in enumerate(drivers_sorted.iterrows()):
    ax.text(row['Combined_Score'] + 0.01, i,
            f"Score: {row['Combined_Score']:.3f}  |  Coef. LR: {row['LR_coef']:+.4f}",
            va='center', fontsize=9)

ax.set_xlabel('Importancia combinada (3 modelos)', fontsize=12)
ax.set_title('Drivers de Satisfacción del Cliente\n'
             'Ranking por importancia + dirección del efecto (coeficiente LR)',
             fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#27ae60', label='Aumenta satisfacción (+)'),
    Patch(facecolor='#c0392b', label='Disminuye satisfacción (-)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11,
          framealpha=0.9)

plt.tight_layout()
plt.show()

print("\nResumen ejecutivo:")
print("=" * 60)
for i, (_, row) in enumerate(drivers_sorted.iterrows(), 1):
    efecto = "AUMENTA" if row['LR_coef'] > 0 else "DISMINUYE"
    print(f"  {i:2d}. {row['Label']:<25s} → {efecto} satisfacción (coef: {row['LR_coef']:+.4f})")

## Resumen

### Modelos entrenados
| Modelo | Enfoque | Fortaleza |
|--------|---------|-----------|
| **Regresión Lineal** | Baseline, coeficientes interpretables | Dirección y magnitud del efecto de cada variable |
| **Random Forest** | Ensemble de árboles, no lineal | Feature importance, captura interacciones |
| **XGBoost/GradientBoosting** | Boosting secuencial | Mayor precisión, permutation importance robusta |

### Hallazgos clave sobre drivers de satisfacción

1. **Drivers principales identificados** por consenso de 3 modelos (ranking combinado de importancias)
2. **Partial Dependence Plots** revelan cómo cada factor afecta la satisfacción de forma no lineal
3. **Análisis por segmento** identifica los grupos de clientes con menor satisfacción (oportunidades de mejora)

### Recomendaciones de negocio

- **Segmentos prioritarios**: Los mapas de calor revelan combinaciones tipo/ingreso/estilo con satisfacción baja - son los segmentos donde intervenir primero
- **Factores controlables**: Variables como consumo, km recorridos y frenadas bruscas son indicadores de la experiencia de conducción que pueden mejorarse con recomendaciones al conductor
- **Factores demográficos**: La edad y el ingreso son factores no controlables, pero permiten personalizar la comunicación y expectativas del cliente
- **Tipo de vehículo**: Las diferencias de satisfacción entre tipos sugieren oportunidades en la experiencia post-venta diferenciada

### Limitaciones
- El target ordinal (1-5) tiene poca granularidad para regresión continua
- R² puede ser bajo porque la satisfacción depende de factores no capturados (servicio post-venta, precio, etc.)
- Los datos simulados pueden no reflejar patrones reales de satisfacción

---

**Siguiente notebook:** [04 - Clasificación: Riesgo de Conductor](04_classification_driver_risk.ipynb) - Pasamos de regresión a clasificación binaria para predecir conductores de alto riesgo.